# pandas-ta-classic Computation Library EDA

Use the Quant fork of pandas-ta-classic for large split technical feature families. This notebook shows direct indicator calls and the existing Quant Warehouse feature-engineering wrapper.

In [1]:
from __future__ import annotations

import importlib
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

def required_library_status(module_name: str) -> pd.DataFrame:
    try:
        module = importlib.import_module(module_name)
        return pd.DataFrame([{
            "module": module_name,
            "required": True,
            "installed": True,
            "version": getattr(module, "__version__", "unknown"),
            "module_file": getattr(module, "__file__", None),
        }])
    except Exception as exc:
        return pd.DataFrame([{
            "module": module_name,
            "required": True,
            "installed": False,
            "version": None,
            "module_file": None,
            "error": f"{type(exc).__name__}: {exc}",
        }])

def synthetic_prices(rows: int = 360, symbol: str = "AAPL") -> pd.DataFrame:
    rng = np.random.default_rng(7)
    dates = pd.date_range("2024-01-02", periods=rows, freq="B")
    drift = 0.00055
    shock = rng.normal(0, 0.015, rows)
    close = 100 * np.exp(np.cumsum(drift + shock))
    open_ = close * (1 + rng.normal(0, 0.003, rows))
    high = np.maximum(open_, close) * (1 + rng.uniform(0.001, 0.02, rows))
    low = np.minimum(open_, close) * (1 - rng.uniform(0.001, 0.02, rows))
    volume = rng.integers(900_000, 4_500_000, rows).astype(float)
    return pd.DataFrame({
        "symbol": symbol,
        "date": dates,
        "open": open_,
        "high": high,
        "low": low,
        "close": close,
        "volume": volume,
    })

prices = synthetic_prices()
prices.head()

,symbol,date,open,high,low,close,volume
0,AAPL,2024-01-02,100.2080,101.2355,98.8738,100.0569,"4,479,129.0000"
1,AAPL,2024-01-03,101.1259,102.2340,100.4497,100.5615,"2,365,734.0000"
2,AAPL,2024-01-04,100.3819,101.9042,99.3952,100.2040,"1,409,205.0000"
3,AAPL,2024-01-05,98.9452,99.8670,97.0239,98.9286,"4,225,356.0000"
4,AAPL,2024-01-08,97.8130,98.6722,97.4943,98.3103,"2,886,674.0000"


## Required Dependency Status

In [2]:
required_library_status("pandas_ta_classic")

,module,required,installed,version,module_file
0,pandas_ta_classic,True,True,0.0.1.dev1017,/home/jlee153232/miniconda3/lib/python3.13/sit...


## Direct Library Usage

This is useful for quick indicator prototyping before promoting a feature into `feature_engineering`. The warehouse wrapper is preferred for reusable feature families because it standardizes prefixes and index shape.

In [3]:
import pandas_ta_classic as ta

work = prices.set_index("date")
features = pd.DataFrame(index=work.index)
features["ta_rsi_14"] = ta.rsi(work["close"], length=14)
macd = ta.macd(work["close"])
if isinstance(macd, pd.DataFrame):
    features = features.join(macd.add_prefix("ta_macd__"))
bbands = ta.bbands(work["close"], length=20)
if isinstance(bbands, pd.DataFrame):
    features = features.join(bbands.add_prefix("ta_bbands__"))
features.tail(8)

,ta_rsi_14,ta_macd__MACD_12_26_9,ta_macd__MACDh_12_26_9,ta_macd__MACDs_12_26_9,ta_bbands__BBL_20_2.0,ta_bbands__BBM_20_2.0,ta_bbands__BBU_20_2.0,ta_bbands__BBB_20_2.0,ta_bbands__BBP_20_2.0
date,,,,,,,,,
2025-05-08,54.8839,0.8922,-0.0118,0.9039,64.6072,68.5566,72.5060,11.5216,0.6266
2025-05-09,53.9810,0.8184,-0.0684,0.8868,64.7670,68.6716,72.5761,11.3716,0.5908
2025-05-12,55.1646,0.7722,-0.0917,0.8639,65.1550,68.8600,72.5650,10.7611,0.6056
2025-05-13,56.7616,0.7551,-0.0871,0.8422,65.8840,69.1183,72.3527,9.3588,0.6350
2025-05-14,55.0991,0.7111,-0.1048,0.8159,66.3653,69.2983,72.2313,8.4649,0.5714
2025-05-15,55.0759,0.6683,-0.1181,0.7864,67.2448,69.5209,71.7970,6.5479,0.5423
2025-05-16,56.4481,0.6475,-0.1112,0.7586,67.9404,69.7010,71.4616,5.0519,0.5759
2025-05-19,56.0888,0.6198,-0.1111,0.7309,68.2353,69.8001,71.3648,4.4835,0.5379


## Quant Warehouse Feature Engineering Usage

The existing module `build_price_ta_classic_feature_families` creates split feature families such as momentum, overlap, candles, cycles, math, and performance. These families can be stored independently and benchmarked against other libraries.

In [4]:
from quant_warehouse.feature_engineering import build_price_ta_classic_feature_families

built_by_family = build_price_ta_classic_feature_families("AAPL", prices)
summary = []
for family_name, built in built_by_family.items():
    summary.append({
        "family": family_name,
        "rows": len(built.df),
        "feature_count": len(built.feature_cols),
        "sample_features": ", ".join(built.feature_cols[:5]),
    })
pd.DataFrame(summary).sort_values("feature_count", ascending=False)

,family,rows,feature_count,sample_features
3,technical_momentum,360,170,"ta_momentum__rsi_14, ta_momentum__macd_12_26_9..."
4,technical_overlap,360,112,"ta_overlap__sma_10, ta_overlap__sma_20, ta_ove..."
0,technical_candles,360,72,"ta_candle__doji_cdl_doji_10_0_1, ta_candle__in..."
2,technical_math,360,25,"ta_math__zscore_20_zs_20, ta_math__zscore_63_z..."
1,technical_cycles,360,11,"ta_cycle__ebsw_40_10, ta_cycle__ht_dcperiod, t..."
5,technical_performance,360,9,"ta_performance__pct_return_1_pctret_1, ta_perf..."


In [5]:
momentum = built_by_family["technical_momentum"].df
selected = [col for col in momentum.columns if "rsi" in col or "macd" in col][:8]
momentum[selected].tail(5)

,,ta_momentum__rsi_14,ta_momentum__macd_12_26_9,ta_momentum__macd_mac_dh_12_26_9,ta_momentum__macd_mac_ds_12_26_9,ta_momentum__macdext_12_26_9,ta_momentum__macdext_macdex_ts_12_26_9,ta_momentum__macdext_macdex_th_12_26_9,ta_momentum__macdfix_9_9
date,symbol,,,,,,,,
2025-05-13,AAPL,56.7616,0.7551,-0.0871,0.8422,0.7551,0.8422,-0.0871,0.7551
2025-05-14,AAPL,55.0991,0.7111,-0.1048,0.8159,0.7111,0.8159,-0.1048,0.7111
2025-05-15,AAPL,55.0759,0.6683,-0.1181,0.7864,0.6683,0.7864,-0.1181,0.6683
2025-05-16,AAPL,56.4481,0.6475,-0.1112,0.7586,0.6475,0.7586,-0.1112,0.6475
2025-05-19,AAPL,56.0888,0.6198,-0.1111,0.7309,0.6198,0.7309,-0.1111,0.6198


## Target Engineering Connection

These indicators are features, not labels. A common workflow is to pair them with forward-return or rank labels from `target_engineering`.

In [6]:
from quant_warehouse.target_engineering import build_forward_return_labels, build_cross_sectional_rank_labels

label_input = prices[["symbol", "date", "close"]]
forward = build_forward_return_labels(label_input, horizons=[5, 20])
ranked = build_cross_sectional_rank_labels(forward)
forward.tail(), ranked.tail()

(    symbol       date  horizon         target_name  target_value
 715   AAPL 2025-05-15       20  forward_return_20d           NaN
 716   AAPL 2025-05-16        5   forward_return_5d           NaN
 717   AAPL 2025-05-16       20  forward_return_20d           NaN
 718   AAPL 2025-05-19        5   forward_return_5d           NaN
 719   AAPL 2025-05-19       20  forward_return_20d           NaN,
     symbol       date  horizon                      target_name  target_value  rank  rank_pct
 715   AAPL 2025-05-15       20  cross_sectional_return_rank_pct           NaN   NaN       NaN
 716   AAPL 2025-05-16        5  cross_sectional_return_rank_pct           NaN   NaN       NaN
 717   AAPL 2025-05-16       20  cross_sectional_return_rank_pct           NaN   NaN       NaN
 718   AAPL 2025-05-19        5  cross_sectional_return_rank_pct           NaN   NaN       NaN
 719   AAPL 2025-05-19       20  cross_sectional_return_rank_pct           NaN   NaN       NaN)

## Notes

- This repo requires the Quant fork: `pandas-ta-classic @ git+https://github.com/quantarb/pandas-ta-classic.git@main`.
- Store pandas-ta-classic outputs as their own feature families; do not overwrite equivalent indicators from vectorized/custom implementations.
- Benchmark speed and signal quality before deciding which implementation to use downstream.